# RAG Pipeline
## Data Ingestion to VectorDB

In [22]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [23]:
from langchain_core.documents import Document
import os

def process_all_texts(text_directory):
    """Process all .txt files in a directory"""
    all_documents = []
    text_dir = Path(text_directory)
    text_files = list(text_dir.glob("**/*.txt"))
    print(f"Found {len(text_files)} text files to process")
    for text_file in text_files:
        print(f"\nProcessing: {text_file.name}")
        try:
            with open(text_file, 'r', encoding='utf-8') as f:
                content = f.read()
            doc = Document(
                page_content=content,
                metadata={
                    'source_file': text_file.name,
                    'file_type': 'txt'
                }
            )
            all_documents.append(doc)
            print(f"  ✓ Loaded text file")
        except Exception as e:
            print(f"  ✗ Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all text files in the data directory
all_text_documents = process_all_texts("../data")

Found 7 text files to process

Processing: ACCOUNT_SERVICE_API_DOCUMENTATION.txt
  ✓ Loaded text file

Processing: ADMIN_SERVICE_API_DOCUMENTATION.txt
  ✓ Loaded text file

Processing: BANK_SERVICE_API_DOCUMENTATION.txt
  ✓ Loaded text file

Processing: FAQ.txt
  ✓ Loaded text file

Processing: ROOT_ADMIN_SERVICE_API_DOCUMENTATION.txt
  ✓ Loaded text file

Processing: USER_GUIDE.txt
  ✓ Loaded text file

Processing: USER_SERVICE_API_DOCUMENTATION.txt
  ✓ Loaded text file

Total documents loaded: 7


In [24]:
all_text_documents

[Document(metadata={'source_file': 'ACCOUNT_SERVICE_API_DOCUMENTATION.txt', 'file_type': 'txt'}, page_content='\n\nACCOUNT-Service\nThe Account Service is a core backend microservice responsible for managing the complete lifecycle of bank accounts in a multi-country, multi-bank environment. It is designed to handle account creation, document management, application status tracking, and administrative approval or rejection workflows for customers in India, USA, and UK. The service enforces strict validation and compliance rules for each country, ensuring that all regulatory and business requirements are met.\n\nKey Features:\n\n- Multi-country support: Implements separate account creation endpoints and request DTOs for India, USA, and UK. Each country has its own validation rules (e.g., Aadhaar/PAN for India, SSN for USA, NIN for UK) and required fields. The service uses a strategy pattern to encapsulate country-specific logic, ensuring compliance with local regulations.\n\n- RESTful AP

In [45]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=800,chunk_overlap=100):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [46]:
# Use a smaller chunk size to avoid exceeding embedding model context length
chunks=split_documents(all_text_documents, chunk_size=800, chunk_overlap=100)
chunks

Split 7 documents into 113 chunks

Example chunk:
Content: ACCOUNT-Service
The Account Service is a core backend microservice responsible for managing the complete lifecycle of bank accounts in a multi-country, multi-bank environment. It is designed to handle...
Metadata: {'source_file': 'ACCOUNT_SERVICE_API_DOCUMENTATION.txt', 'file_type': 'txt'}


[Document(metadata={'source_file': 'ACCOUNT_SERVICE_API_DOCUMENTATION.txt', 'file_type': 'txt'}, page_content='ACCOUNT-Service\nThe Account Service is a core backend microservice responsible for managing the complete lifecycle of bank accounts in a multi-country, multi-bank environment. It is designed to handle account creation, document management, application status tracking, and administrative approval or rejection workflows for customers in India, USA, and UK. The service enforces strict validation and compliance rules for each country, ensuring that all regulatory and business requirements are met.\n\nKey Features:'),
 Document(metadata={'source_file': 'ACCOUNT_SERVICE_API_DOCUMENTATION.txt', 'file_type': 'txt'}, page_content='Key Features:\n\n- Multi-country support: Implements separate account creation endpoints and request DTOs for India, USA, and UK. Each country has its own validation rules (e.g., Aadhaar/PAN for India, SSN for USA, NIN for UK) and required fields. The servic

### Embedding and VectoStoreDB

In [47]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [48]:
import requests
import numpy as np

class EmbeddingManager:
    """Handles document embedding generation using Ollama API"""
    
    def __init__(self, ollama_url: str = "http://localhost:11434/api/embeddings", model: str = "all-minilm"):
        """
        Initialize the embedding manager for Ollama
        
        Args:
            ollama_url: URL of the Ollama embeddings API endpoint
            model: Name of the embedding model to use
        """
        self.ollama_url = ollama_url
        self.model = model
    
    def generate_embeddings(self, texts: list) -> np.ndarray:
        """
        Generate embeddings for a list of texts using Ollama API
        
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        embeddings = []
        for text in texts:
            payload = {
                "model": self.model,
                "prompt": text
            }
            response = requests.post(self.ollama_url, json=payload)
            try:
                response.raise_for_status()
            except requests.HTTPError as e:
                print(f"HTTPError: {e}")
                print(f"Status code: {response.status_code}")
                print(f"Response content: {response.text}")
                raise
            data = response.json()
            if "embedding" not in data:
                raise ValueError(f"No embedding returned for text: {text}")
            embeddings.append(data["embedding"])
        embeddings = np.array(embeddings)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

In [49]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "all_text_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Text document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: all_text_documents
Existing documents in collection: 573


In [51]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generated embeddings with shape: (113, 384)
Adding 113 documents to vector store...
Successfully added 113 documents to vector store
Total documents in collection: 686


In [36]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [37]:
rag_retriever

In [38]:
rag_retriever.retrieve("In How many ways can I open a bank account?")

Retrieving documents for query: 'In How many ways can I open a bank account?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_57c33352_343',
  'content': '6. Create a New Bank Account\nRoute: /createaccount\n• Go to the “Create Account” page (/createaccount).\n• Complete each step of the multi-step form: select branch, enter personal, educational, income, and nominee details, and upload documents.',
  'metadata': {'file_type': 'txt',
   'content_length': 242,
   'source_file': 'USER_GUIDE.txt',
   'doc_index': 343},
  'similarity_score': 0.1635669469833374,
  'distance': 0.8364330530166626,
  'rank': 1},
 {'id': 'doc_584ba175_164',
  'content': '6. Create a New Bank Account\nRoute: /createaccount\n• Go to the “Create Account” page (/createaccount).\n• Complete each step of the multi-step form: select branch, enter personal, educational, income, and nominee details, and upload documents.\n• Each step validates your input before proceeding.\n• Submit the form to create your account. Success or error messages are shown as appropriate.',
  'metadata': {'source_file': 'USER_GUIDE.txt',
   'doc_index':

In [39]:
rag_retriever.retrieve("In How many countries can I open a bank account?")

Retrieving documents for query: 'In How many countries can I open a bank account?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ce4c1a88_124',
  'content': 'Important Points:\n    - country must be one of: india, usa, uk.\n    - Returns valid account types for the specified country.',
  'metadata': {'content_length': 124,
   'doc_index': 124,
   'source_file': 'ACCOUNT_SERVICE_API_DOCUMENTATION.txt',
   'file_type': 'txt'},
  'similarity_score': 0.28884053230285645,
  'distance': 0.7111594676971436,
  'rank': 1},
 {'id': 'doc_f292b249_62',
  'content': 'Important Points:\n    - country must be one of: india, usa, uk.\n    - Returns valid account types for the specified country.',
  'metadata': {'doc_index': 62,
   'source_file': 'ACCOUNT_SERVICE_API_DOCUMENTATION.txt',
   'content_length': 124,
   'file_type': 'txt'},
  'similarity_score': 0.28884053230285645,
  'distance': 0.7111594676971436,
  'rank': 2},
 {'id': 'doc_6ed73424_63',
  'content': '-----------------------------\n  Important Points: Required Documents for Account Creation\n\n  For account creation (all countries), the following docum

RAG Pipeline- VectorDB To LLM Output Generation

In [40]:
import json

import requests

def ollama_rag_response(
    query, 
    rag_retriever, 
    ollama_url="http://localhost:11434/api/generate", 
    model_name="llama3.2:3b", 
    top_k=5
):
    # Step 1: Retrieve relevant context from vector DB
    retrieved_docs = rag_retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else ""
    if not context:
        return "No relevant context found to answer the question."

    # Step 2: Prepare prompt for LLM
    print("calling llm")
    prompt = f"""Context:\n{context}\n\nQuestion: {query}\nAnswer:"""
    print("loading data")
    # Step 3: Call Ollama LLM API (streaming)
    payload = {
        "model": model_name,
        "prompt": prompt
    }
    print("payload prepared, making API call...")
    response = requests.post(ollama_url, json=payload, stream=True)
    response.raise_for_status()
    answer = ""
    for line in response.iter_lines():
        if line:
            try:
                data = json.loads(line.decode("utf-8"))
                answer += data.get("response", "")
            except Exception:
                continue
    return answer if answer else "No response from LLM."

In [41]:
question = "On which countries can I open a bank account?"
response = ollama_rag_response(question, rag_retriever, model_name="llama3.2:3b")

Retrieving documents for query: 'On which countries can I open a bank account?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
calling llm
loading data
payload prepared, making API call...


In [42]:
print("Question:", question)
print("LLM Response:", response)

Question: On which countries can I open a bank account?
LLM Response: You can open a bank account in one of the following countries:

1. India
2. USA
3. UK


In [43]:
question = "What Documents Do I need to open a bank account?"
response = ollama_rag_response(question, rag_retriever, model_name="llama3.2:3b")

Retrieving documents for query: 'What Documents Do I need to open a bank account?'
Top K: 5, Score threshold: 0.0
Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
calling llm
loading data
payload prepared, making API call...


In [44]:
print("Question:", question)
print("LLM Response:", response)

Question: What Documents Do I need to open a bank account?
LLM Response: To open a bank account, you need the following documents:

1. Government-issued identity proof (e.g., Aadhaar for India, SSN for USA, NIN for UK)
2. Proof of address (e.g., utility bill, rental agreement, bank statement)
3. Proof of income (e.g., salary slip, tax return, employment letter)
4. Recent passport-size photograph
